# Module 11: Confounders, Mediators and Colliders

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Three variables can sit beside a treatment and an outcome, they look alike in
a spreadsheet, and the correct handling is different for each.

- **Confounder:** control for it
- **Mediator:** do **not** control for it
- **Collider:** do **not** control for it, and controlling for it can flip a
  sign

Getting the third one wrong is the least known and the most damaging.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

keep = [a for a in TRAINED if a != "A007"]
d = f[f["agency_id"].isin(keep + COMPARISON)].copy()
d["lo"] = np.log(d["n_arrests"])
d["tr"] = d["agency_id"].isin(keep).astype(float)
d["settled"] = ((d["tr"] == 1) & (d["period"] == "after")).astype(float)
d["phase"] = ((d["tr"] == 1) & (d["period"] == "phase")).astype(float)

pct = lambda b: 100 * (np.exp(b) - 1)


def effect(outcome, extra="", offset=None):
    form = f"{outcome} ~ C(agency_id) + C(year_month) + settled + phase{extra}"
    z = smf.glm(form, d, family=sm.families.Poisson(), offset=offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. Telling them apart

The distinction is about **which way the arrows point**, and no amount of
looking at the data reveals it. It comes from knowing how the world works.

| Shape | Arrows | Rule |
|---|---|---|
| **Confounder** | it → treatment, it → outcome | control for it |
| **Mediator** | treatment → it → outcome | leave it alone |
| **Collider** | treatment → it, outcome → it | leave it alone |

A confounder creates a spurious association. A mediator carries the real one.
A collider creates a spurious association **only if you condition on it**.

## 3. A confounder: the statewide decline

It moved both. Agencies were selected during a period when everything was
falling, and everything kept falling. Controlling for it is what the month
fixed effects do.

In [ ]:
e0 = effect("n_uof", extra="", offset=d["lo"])
z_nofe = smf.glm("n_uof ~ C(agency_id) + settled + phase", d,
                 family=sm.families.Poisson(), offset=d["lo"]).fit()
print(f"  with month effects, which absorb the statewide decline: "
      f"{e0[0]:+.1f}%")
print(f"  without them:                                           "
      f"{pct(z_nofe.params['settled']):+.1f}%")
print(f"  the truth:                                              {TRUTH:+.1f}%")

Failing to control for the confounder more than doubles the estimate. This is
the case everyone knows.

## 4. A mediator: the denominator

The outcome is use of force **per arrest**. If the training changed how many
arrests officers made, then arrests sit on the path from the program to the
measured rate, and dividing by them removes part of what is being measured.

So the first question is whether the program moved the denominator.

In [ ]:
for label, outcome, off in [("use of force per arrest", "n_uof", d["lo"]),
                            ("use of force, raw count", "n_uof", None),
                            ("arrests", "n_arrests", None),
                            ("calls for service", "total_cfs", None)]:
    e, lo, hi, _ = effect(outcome, offset=off)
    flag = "" if lo < 0 < hi else "   distinguishable from zero"
    print(f"  {label:26s} {e:+7.2f}%  [{lo:+6.2f}, {hi:+6.2f}]{flag}")

**Arrests did not move**, so the denominator is safe and the rate is not a
mediated measure. The raw count and the rate agree to within a twentieth of a
point, which is the check.

**Calls for service moved by 0.42 percent, and the interval excludes zero.**
That is a useful embarrassment. With 964 agency months and tens of thousands
of calls, a difference far too small to matter is easily distinguishable from
zero. Statistical detectability and substantive size are different things,
and the interval is what lets you tell.

## 5. Controlling for a mediator, deliberately

To see the damage, control for something the program plausibly acts through.
Here that is arrests: not as an offset with its coefficient fixed at 1, but
as a free covariate that the model can use to explain away the effect.

In [ ]:
e_off = effect("n_uof", offset=d["lo"])
e_free = effect("n_uof", extra=" + lo")
print(f"  arrests as an offset, coefficient fixed at 1: {e_off[0]:+.2f}%")
print(f"  arrests as a free covariate:                  {e_free[0]:+.2f}%"
      f"   its coefficient {e_free[3].params['lo']:.2f}")

Here the two agree, because the program genuinely did not change arrests, and
the freely estimated coefficient comes out at 1.02, which is what the offset
assumes.

**That agreement is a finding, not a formality.** If the program had reduced
use of force partly by reducing arrests, the free covariate would absorb that
route and the estimate would shrink. Running both and reporting that they
match is how you establish the denominator is not a mediator.

## 6. A collider, constructed

Colliders are hard to demonstrate with observational data because they
require conditioning on something caused by both. Here is one built on
purpose, with real numbers, so the mechanism is visible.

Suppose the state reviews an agency if it **either** took the training **or**
had a high use of force rate afterwards. The review is caused by both.

In [ ]:
after = d[d["period"] == "after"].groupby("agency_id").apply(
    lambda g: 100 * g["n_uof"].sum() / g["n_arrests"].sum())
tab = pd.DataFrame({"trained": [1 if a in keep else 0 for a in after.index],
                    "rate after": after.round(2)})
tab["reviewed"] = ((tab["trained"] == 1) | (tab["rate after"] > 2.5)).astype(int)
tab.index = [NAME[a] for a in after.index]
display(tab)

all_c = np.corrcoef(tab["trained"], tab["rate after"])[0, 1]
sub = tab[tab["reviewed"] == 1]
sub_c = np.corrcoef(sub["trained"], sub["rate after"])[0, 1]
print(f"\n  among all agencies:      correlation = {all_c:+.2f}")
print(f"  among reviewed agencies: correlation = {sub_c:+.2f}")

**The sign flips.** Across all agencies, trained agencies have higher rates
after the program, because they started far higher. Among reviewed agencies
only, trained agencies have *lower* rates.

Nothing changed about any agency. The only difference is which rows were
looked at, and the rule for looking was caused by both variables.

An untrained agency only appears in the reviewed group if its rate was high;
a trained agency appears regardless. So conditioning on review selects the
worst untrained agencies against all trained agencies, and manufactures a
negative association.

**This is why "the sample was restricted to agencies under review" or "only
agencies that responded to the survey were analysed" needs scrutiny.** Any filter that the
treatment and the outcome both influence is a collider.

## 7. The decision rule

For any variable, ask two questions in order.

1. **Could it have been affected by the program?** If yes, it is not a
   confounder. Leave it out, or handle it as a mediator in a separate
   analysis that is labelled as such.
2. **Could it have affected who got the program, and separately the
   outcome?** If yes, it is a confounder. Control for it.

Anything that is neither can be left out without cost. Anything that is both
is a hard problem and belongs in the limitations.

**A variable measured after the treatment started is guilty until proven
innocent.** That single rule prevents most mediator and collider mistakes.

## Exercise

Test the guilty until proven innocent rule on a variable that looks
completely harmless: the number of sworn officers.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    d2 = d.copy()
    d2["log_sworn"] = np.log(d2["sworn_officers"])
    e1 = effect("n_uof", offset=d["lo"])
    z = smf.glm("n_uof ~ C(agency_id) + C(year_month) + log_sworn + settled + phase",
                d2, family=sm.families.Poisson(), offset=d2["lo"]).fit()
    lo, hi = z.conf_int().loc["settled"]
    zs = smf.glm("sworn_officers ~ C(agency_id) + C(year_month) + settled + phase",
                 d, family=sm.families.Poisson()).fit()
    slo, shi = zs.conf_int().loc["settled"]
    print(f"  did the program change staffing?  "
          f"{pct(zs.params['settled']):+.2f}%  [{pct(slo):+.2f}, {pct(shi):+.2f}]")
    print(f"\n  estimate without controlling for staffing: {e1[0]:+.2f}%")
    print(f"  estimate controlling for staffing:         "
          f"{pct(z.params['settled']):+.2f}%  [{pct(lo):+.2f}, {pct(hi):+.2f}]")
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Staffing did not change and controlling for it changes nothing, so on this
dataset it is harmless.

**The point is the order of the two checks, not the answer.** The first cell
asks whether the program moved the variable, and only because the answer is
no is it safe to put it in the model. Had staffing risen at the trained
agencies, adding it would have been controlling for a consequence of the
program and the estimate would no longer mean what it says.

Note also that with agency fixed effects already in the model, a variable
that barely moves within an agency contributes almost nothing. A good deal of
covariate adjustment in panel work is doing no work at all, which is not a
problem, but it is worth knowing before defending it in a seminar.

</details>

---

**Next:** [Module 12: Placebo Tests](Module_12_Placebo_Tests.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*